# LeetCode 1466 — Reorder Routes to Make All Paths Lead to the City Zero bằng Python

Link đề: https://leetcode.com/problems/reorder-routes-to-make-all-paths-lead-to-the-city-zero/ · Độ khó: **Medium** · Chủ đề: Depth-First Search, Breadth-First Search, Graph Theory

Skeleton LeetCode:

```python
class Solution:
    def minReorder(self, n: int, connections: list[list[int]]) -> int:
        ...
```

## 1. Mục tiêu học

1. Nhận ra mạng `n` thành phố và `n − 1` con đường liên thông là một **cây**, và trong cây giữa hai đỉnh có **đúng một đường đi**.
2. Hiểu điều kiện "mọi thành phố tới được 0" ⇔ **mọi cạnh phải hướng về phía gốc 0**, nên đáp án = số cạnh đang hướng **ra xa** gốc.
3. Kỹ thuật **coi đồ thị là vô hướng nhưng ghi nhớ chiều gốc** (cờ `cost = 1` cho cạnh thuận, `0` cho cạnh ngược) rồi DFS/BFS từ 0.
4. Cài DFS lặp bằng stack để tránh **giới hạn đệ quy** khi cây là một chuỗi dài `5·10^4`.
5. Mở rộng: **rerooting** — tính đáp án cho **mọi** thành phố làm thủ đô trong `O(n)`.

## 2. Đề bài & ví dụ

Có `n` thành phố `0 .. n−1` và `n − 1` con đường sao cho giữa hai thành phố bất kỳ có đúng một cách đi (mạng là một cây). Mỗi đường là **một chiều**: `connections[i] = [a_i, b_i]` là đường đi **từ `a_i` tới `b_i`**.

Thủ đô là thành phố `0`. Hãy **đổi chiều ít con đường nhất** để mọi thành phố đều đi được tới `0`. Trả về số đường cần đổi chiều. Đề đảm bảo luôn làm được.

**Ví dụ 1:** `n = 6, connections = [[0,1],[1,3],[2,3],[4,0],[4,5]]` → `3` (đổi `0→1`, `1→3`, `4→5`).

**Ví dụ 2:** `n = 5, connections = [[1,0],[1,2],[3,2],[3,4]]` → `2` (đổi `1→2`, `3→4`).

**Ví dụ 3:** `n = 3, connections = [[1,0],[2,0]]` → `0`.

**Ràng buộc:**
- `2 <= n <= 5 * 10^4`
- `connections.length == n − 1`, `connections[i].length == 2`
- `0 <= a_i, b_i <= n − 1`, `a_i != b_i`

In [1]:
import random
import sys
import time
from collections import deque
from itertools import product
from typing import List

examples = [
    (6, [[0, 1], [1, 3], [2, 3], [4, 0], [4, 5]], 3),
    (5, [[1, 0], [1, 2], [3, 2], [3, 4]], 2),
    (3, [[1, 0], [2, 0]], 0),
]
for n, conn, want in examples:
    print(f"n={n}, connections={conn} -> kỳ vọng {want}")

n=6, connections=[[0, 1], [1, 3], [2, 3], [4, 0], [4, 5]] -> kỳ vọng 3
n=5, connections=[[1, 0], [1, 2], [3, 2], [3, 4]] -> kỳ vọng 2
n=3, connections=[[1, 0], [2, 0]] -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1 Cây và đường đi duy nhất

Đồ thị vô hướng liên thông có `n` đỉnh, `n − 1` cạnh là **cây**: không có chu trình, giữa hai đỉnh có đúng một đường đi. Nếu chọn `0` làm **gốc**, mỗi đỉnh `v ≠ 0` có đúng một **cha** (đỉnh kề trên đường về 0).

```
Ví dụ 1 (bỏ chiều):      0
                        / \
                       1   4
                       |   |
                       3   5
                       |
                       2
```

In [2]:
def parents_from_root(n, connections, root=0):
    adj = [[] for _ in range(n)]
    for a, b in connections:
        adj[a].append(b)
        adj[b].append(a)
    parent = [-1] * n
    parent[root] = root
    stack = [root]
    while stack:
        u = stack.pop()
        for v in adj[u]:
            if parent[v] == -1:
                parent[v] = u
                stack.append(v)
    return parent

print("cha của từng đỉnh (ví dụ 1):", parents_from_root(*examples[0][:2]))

cha của từng đỉnh (ví dụ 1): [0, 0, 3, 1, 0, 4]


### 3.2 Danh sách kề vô hướng có ghi chiều gốc

Với cạnh `a → b`, thêm vào **cả hai** đầu, kèm cờ cho biết đi theo hướng này có **trùng chiều gốc** không:
- `adj[a].append((b, 1))` — đi `a → b` đúng chiều đường; nếu ta đang đi **ra xa gốc** theo hướng này thì đường đang hướng ra xa ⇒ **phải đổi** (chi phí 1).
- `adj[b].append((a, 0))` — đi `b → a` ngược chiều đường; đường thật đang hướng `a → b`, tức hướng **về** phía ta vừa đi từ đó ⇒ không cần đổi.

In [3]:
def build_adj(n, connections):
    adj = [[] for _ in range(n)]
    for a, b in connections:
        adj[a].append((b, 1))     # cạnh gốc a -> b
        adj[b].append((a, 0))     # chiều ngược, "ảo"
    return adj

for u, lst in enumerate(build_adj(*examples[0][:2])):
    print(u, lst)

0 [(1, 1), (4, 0)]
1 [(0, 0), (3, 1)]
2 [(3, 1)]
3 [(1, 0), (2, 0)]
4 [(0, 1), (5, 1)]
5 [(4, 0)]


### 3.3 DFS lặp với stack và mảng `visited`

Đồ thị vô hướng: từ `u` sang `v` rồi từ `v` có thể quay lại `u`. Cần `visited` (hoặc nhớ `parent`) để không đi vòng vô hạn.

In [4]:
def dfs_order(adj, s):
    seen = [False] * len(adj)
    seen[s] = True
    stack, order = [s], []
    while stack:
        u = stack.pop()
        order.append(u)
        for v, _ in adj[u]:
            if not seen[v]:
                seen[v] = True
                stack.append(v)
    return order

print("thứ tự DFS từ 0:", dfs_order(build_adj(*examples[0][:2]), 0))

thứ tự DFS từ 0: [0, 4, 5, 1, 3, 2]


### 3.4 Giới hạn đệ quy của Python

Mặc định `sys.getrecursionlimit()` khoảng `1000`. DFS đệ quy trên một chuỗi `5·10^4` đỉnh sẽ vượt giới hạn. Có thể tăng giới hạn, nhưng an toàn hơn là viết **DFS lặp** hoặc **BFS**.

In [5]:
print("giới hạn đệ quy hiện tại:", sys.getrecursionlimit())

def depth(k):
    return 0 if k == 0 else 1 + depth(k - 1)

try:
    depth(5000)
except RecursionError as e:
    print("RecursionError:", str(e)[:50])

giới hạn đệ quy hiện tại: 1000
RecursionError: maximum recursion depth exceeded


### 3.5 `itertools.product` để thử mọi tổ hợp

`product([0, 1], repeat=m)` sinh mọi dãy nhị phân độ dài `m` — dùng cho brute force "thử mọi cách đổi chiều".

In [6]:
print(list(product([0, 1], repeat=3)))

[(0, 0, 0), (0, 0, 1), (0, 1, 0), (0, 1, 1), (1, 0, 0), (1, 0, 1), (1, 1, 0), (1, 1, 1)]


## 4. Ý tưởng nền tảng / trực giác

Treo cây ở gốc 0. Để thành phố `v` đi được về 0, nó phải đi lên cha, rồi cha của cha, … ⇒ **mọi cạnh trên đường `v → 0` phải hướng lên (về gốc)**. Áp cho mọi `v` ⇒ **mọi cạnh của cây phải hướng về gốc**.

Tính tay ví dụ 1 (cha: `1→0`, `4→0`, `3→1`, `5→4`, `2→3` là chiều cần có):

| cạnh gốc | chiều cần có (con → cha) | hướng ra xa gốc? | đổi? |
|---|---|---|---|
| 0 → 1 | 1 → 0 | ✔ | 1 |
| 1 → 3 | 3 → 1 | ✔ | 1 |
| 2 → 3 | 2 → 3 | ✘ | 0 |
| 4 → 0 | 4 → 0 | ✘ | 0 |
| 4 → 5 | 5 → 4 | ✔ | 1 |

Tổng `3`. Mỗi cạnh được quyết định **độc lập**: cạnh `cha → con` phải đổi, cạnh `con → cha` giữ nguyên.

In [7]:
n, conn, want = examples[0]
par = parents_from_root(n, conn)
for a, b in conn:
    away = par[b] == a                 # a là cha của b -> cạnh a->b hướng ra xa gốc
    print(f"{a}->{b}: {'đổi' if away else 'giữ'}")
print("tổng:", sum(par[b] == a for a, b in conn))

0->1: đổi
1->3: đổi
2->3: giữ
4->0: giữ
4->5: đổi
tổng: 3


## 5. Cách 1 — Brute Force / baseline: thử mọi cách đổi chiều

**Ý tưởng:** có `n − 1` cạnh, mỗi cạnh giữ hoặc đổi ⇒ `2^(n−1)` cấu hình. Với mỗi cấu hình, kiểm tra mọi thành phố tới được 0 (BFS ngược từ 0 trên đồ thị đảo chiều), lấy số lần đổi nhỏ nhất.

Chỉ chạy được với `n` rất nhỏ (≈ 15), nhưng **không dựa vào bất kỳ suy luận nào** ⇒ là chuẩn đối chiếu tin cậy.

In [8]:
def all_reach_zero(n, edges):
    # đảo chiều mọi cạnh rồi BFS từ 0: ai được thăm thì tới được 0 trong đồ thị gốc
    radj = [[] for _ in range(n)]
    for a, b in edges:
        radj[b].append(a)
    seen = {0}
    q = deque([0])
    while q:
        u = q.popleft()
        for v in radj[u]:
            if v not in seen:
                seen.add(v)
                q.append(v)
    return len(seen) == n

def min_reorder_brute(n, connections):
    best = None
    for flips in product([0, 1], repeat=len(connections)):
        edges = [(b, a) if f else (a, b) for (a, b), f in zip(connections, flips)]
        if all_reach_zero(n, edges):
            cost = sum(flips)
            if best is None or cost < best:
                best = cost
    return best

for n, conn, want in examples:
    got = min_reorder_brute(n, conn)
    print(got, want, "✔" if got == want else "✘")

3 3 ✔
2 2 ✔
0 0 ✔


**Độ phức tạp:** `O(2^(n−1) · n)`. Với `n = 5·10^4` là bất khả thi.

**Nút thắt:** ta thử cả những cấu hình vô lý, trong khi theo mục 4, chiều của mỗi cạnh đã **bị ép buộc** (phải hướng về gốc). Chỉ cần biết với mỗi cạnh, đầu nào gần gốc hơn.

In [9]:
for n in (8, 12, 15):
    rng = random.Random(n)
    conn = []
    for v in range(1, n):
        p = rng.randrange(v)
        conn.append([p, v] if rng.random() < 0.5 else [v, p])
    t = time.perf_counter()
    r = min_reorder_brute(n, conn)
    print(f"n={n:2d}: {2 ** (n - 1):6d} cấu hình, đáp án={r}, {time.perf_counter() - t:.3f}s")

n= 8:    128 cấu hình, đáp án=7, 0.000s
n=12:   2048 cấu hình, đáp án=7, 0.009s
n=15:  16384 cấu hình, đáp án=11, 0.064s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** đáp án = số cạnh có dạng `cha → con` khi treo cây ở gốc 0. Không có lựa chọn nào khác, nên không cần tối ưu gì — chỉ cần **phân loại** từng cạnh.

**Cách phân loại:** đi từ 0 ra ngoài trên đồ thị **vô hướng**. Mỗi khi đi qua cạnh từ `u` (đã thăm, gần gốc) sang `v` (chưa thăm, xa gốc):
- **Nếu** cạnh gốc là `u → v` (đi cùng chiều cạnh) **thì** nó hướng ra xa gốc ⇒ `ans += 1`.
- **Nếu** cạnh gốc là `v → u` **thì** nó đã hướng về gốc ⇒ không cộng.

Ghi cờ `1` vào hướng thật và `0` vào hướng ảo ở danh sách kề, thì chỉ cần `ans += cost`.

**Invariant:** khi `v` được thăm lần đầu từ `u`, `u` là cha của `v` trong cây treo ở 0; mỗi cạnh được xét đúng một lần theo hướng cha → con.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — "Lan dần" tập thành phố đã về được 0 (quét cạnh lặp lại)

Trực quan nhất: giữ tập `ok` = các thành phố đã chắc chắn về được 0 (ban đầu `{0}`). Quét các cạnh chưa xử lý:
- cạnh `a → b` với `b ∈ ok`, `a ∉ ok`: `a` về được qua `b` ⇒ thêm `a`, không tốn.
- cạnh `a → b` với `a ∈ ok`, `b ∉ ok`: phải đổi thành `b → a` ⇒ thêm `b`, `ans += 1`.
- cạnh mà cả hai đầu chưa thuộc `ok`: để vòng sau.

Lặp đến khi hết cạnh. Đúng nhưng có thể cần nhiều vòng quét (cây chuỗi liệt kê ngược ⇒ `O(n²)`).

In [10]:
def min_reorder_sweep(n, connections):
    ok = [False] * n
    ok[0] = True
    pending = list(connections)
    ans = rounds = 0
    while pending:
        rounds += 1
        rest = []
        for a, b in pending:
            if ok[b]:
                ok[a] = True
            elif ok[a]:
                ok[b] = True
                ans += 1
            else:
                rest.append((a, b))
        pending = rest
    return ans

for n, conn, want in examples:
    print(min_reorder_sweep(n, conn), want)

3 3
2 2
0 0


Code nộp LeetCode (Cách 2 — chạy được với dữ liệu thường, nhưng xấu nhất `O(n²)`):

In [11]:
class Solution:
    def minReorder(self, n: int, connections: List[List[int]]) -> int:
        ok = [False] * n
        ok[0] = True
        pending = connections
        ans = 0
        while pending:
            rest = []
            for a, b in pending:
                if ok[b]:
                    ok[a] = True
                elif ok[a]:
                    ok[b] = True
                    ans += 1
                else:
                    rest.append((a, b))
            pending = rest
        return ans

print([Solution().minReorder(n, c) for n, c, _ in examples])

[3, 2, 0]


Minh họa trường hợp xấu: chuỗi `0 – 1 – 2 – … – (n−1)` nhưng các cạnh được liệt kê **từ xa về gần** ⇒ mỗi vòng chỉ xử lý được 1 cạnh.

In [12]:
for n in (500, 1000, 2000):
    conn = [[i + 1, i] for i in reversed(range(n - 1))]    # (n-1 -> n-2), ..., (1 -> 0)
    t = time.perf_counter()
    r = min_reorder_sweep(n, conn)
    print(f"n={n}: đáp án={r}, {time.perf_counter() - t:.3f}s  (gấp đôi n -> ~gấp 4 thời gian)")

n=500: đáp án=0, 0.008s  (gấp đôi n -> ~gấp 4 thời gian)
n=1000: đáp án=0, 0.025s  (gấp đôi n -> ~gấp 4 thời gian)


n=2000: đáp án=0, 0.095s  (gấp đôi n -> ~gấp 4 thời gian)


### 7.2 Cách 3 — DFS từ 0 với cờ chiều (lặp bằng stack)

**Từng đoạn:**
1. `adj[a].append((b, 1))`, `adj[b].append((a, 0))`.
2. Stack bắt đầu bằng `0`, `seen[0] = True`.
3. Lấy `u`, với mỗi `(v, cost)` chưa thăm: `ans += cost`, đánh dấu, đẩy `v`.

In [13]:
def min_reorder_dfs(n, connections):
    adj = build_adj(n, connections)
    seen = [False] * n
    seen[0] = True
    stack = [0]
    ans = 0
    while stack:
        u = stack.pop()
        for v, cost in adj[u]:
            if not seen[v]:
                seen[v] = True
                ans += cost
                stack.append(v)
    return ans

for n, conn, want in examples:
    print(min_reorder_dfs(n, conn), want)

3 3
2 2
0 0


Code nộp LeetCode (Cách 3):

In [14]:
class Solution:
    def minReorder(self, n: int, connections: List[List[int]]) -> int:
        adj = [[] for _ in range(n)]
        for a, b in connections:
            adj[a].append((b, 1))
            adj[b].append((a, 0))
        seen = [False] * n
        seen[0] = True
        stack = [0]
        ans = 0
        while stack:
            u = stack.pop()
            for v, cost in adj[u]:
                if not seen[v]:
                    seen[v] = True
                    ans += cost
                    stack.append(v)
        return ans

print([Solution().minReorder(n, c) for n, c, _ in examples])

[3, 2, 0]


**Độ phức tạp:** `O(n)` thời gian (mỗi cạnh xét 2 lần), `O(n)` bộ nhớ.

### 7.3 Cách 4 — BFS tính khoảng cách tới 0, rồi so sánh hai đầu mỗi cạnh

**Từng đoạn:**
1. BFS trên đồ thị vô hướng từ 0, được `dist[v]` (số cạnh tới 0).
2. Trong cây, hai đầu của một cạnh luôn có `dist` chênh nhau đúng 1; đầu có `dist` nhỏ hơn là **cha**.
3. Cạnh `a → b` cần đổi ⇔ `dist[a] < dist[b]` (đi từ gần ra xa).

Cách này tách bạch hai bước "tìm cấu trúc cây" và "đếm", dễ mở rộng (ví dụ in ra danh sách cạnh cần đổi).

In [15]:
def min_reorder_bfs_dist(n, connections):
    adj = [[] for _ in range(n)]
    for a, b in connections:
        adj[a].append(b)
        adj[b].append(a)
    dist = [-1] * n
    dist[0] = 0
    q = deque([0])
    while q:
        u = q.popleft()
        for v in adj[u]:
            if dist[v] == -1:
                dist[v] = dist[u] + 1
                q.append(v)
    return sum(dist[a] < dist[b] for a, b in connections)

for n, conn, want in examples:
    print(min_reorder_bfs_dist(n, conn), want)

3 3
2 2
0 0


Code nộp LeetCode (Cách 4):

In [16]:
class Solution:
    def minReorder(self, n: int, connections: List[List[int]]) -> int:
        adj = [[] for _ in range(n)]
        for a, b in connections:
            adj[a].append(b)
            adj[b].append(a)
        dist = [-1] * n
        dist[0] = 0
        q = deque([0])
        while q:
            u = q.popleft()
            for v in adj[u]:
                if dist[v] == -1:
                    dist[v] = dist[u] + 1
                    q.append(v)
        return sum(dist[a] < dist[b] for a, b in connections)

print([Solution().minReorder(n, c) for n, c, _ in examples])

[3, 2, 0]


**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ.

## 8. TRACE từng bước

Trace Cách 3 (DFS lặp) trên ví dụ 1. Danh sách kề (đỉnh, cost): `0: (1,1),(4,0)`; `1: (0,0),(3,1)`; `3: (1,0),(2,0)`; `4: (0,1),(5,1)`; …

| lấy u | xét (v, cost) | v mới? | ans | stack sau |
|---|---|---|---|---|
| 0 | (1,1) | ✔ | 1 | [1] |
| 0 | (4,0) | ✔ | 1 | [1, 4] |
| 4 | (0,1) | ✘ | 1 | |
| 4 | (5,1) | ✔ | 2 | [1, 5] |
| 5 | (4,0) | ✘ | 2 | [1] |
| 1 | (0,0) ✘, (3,1) ✔ | | 3 | [3] |
| 3 | (1,0) ✘, (2,0) ✔ | | 3 | [2] |
| 2 | (3,1) ✘ | | 3 | [] |

In [17]:
def trace_dfs(n, connections):
    adj = build_adj(n, connections)
    seen = [False] * n
    seen[0] = True
    stack = [0]
    ans = 0
    while stack:
        u = stack.pop()
        print(f"lấy u={u}")
        for v, cost in adj[u]:
            if not seen[v]:
                seen[v] = True
                ans += cost
                stack.append(v)
                kind = "cạnh gốc u->v, ĐỔI" if cost else "cạnh gốc v->u, giữ"
                print(f"   tới v={v} ({kind}) ans={ans} stack={stack}")
            else:
                print(f"   v={v} đã thăm, bỏ qua")
    print("đáp án:", ans)
    return ans

trace_dfs(*examples[0][:2])

lấy u=0
   tới v=1 (cạnh gốc u->v, ĐỔI) ans=1 stack=[1]
   tới v=4 (cạnh gốc v->u, giữ) ans=1 stack=[1, 4]
lấy u=4
   v=0 đã thăm, bỏ qua
   tới v=5 (cạnh gốc u->v, ĐỔI) ans=2 stack=[1, 5]
lấy u=5
   v=4 đã thăm, bỏ qua
lấy u=1
   v=0 đã thăm, bỏ qua
   tới v=3 (cạnh gốc u->v, ĐỔI) ans=3 stack=[3]
lấy u=3
   v=1 đã thăm, bỏ qua
   tới v=2 (cạnh gốc v->u, giữ) ans=3 stack=[2]
lấy u=2
   v=3 đã thăm, bỏ qua
đáp án: 3


3

In [18]:
def trace_bfs_dist(n, connections):
    adj = [[] for _ in range(n)]
    for a, b in connections:
        adj[a].append(b); adj[b].append(a)
    dist = [-1] * n; dist[0] = 0
    q = deque([0])
    while q:
        u = q.popleft()
        for v in adj[u]:
            if dist[v] == -1:
                dist[v] = dist[u] + 1
                q.append(v)
    print("dist =", dist)
    ans = 0
    for a, b in connections:
        flip = dist[a] < dist[b]
        ans += flip
        print(f"  cạnh {a}->{b}: dist {dist[a]} vs {dist[b]} -> {'ĐỔI' if flip else 'giữ'} | ans={ans}")
    return ans

_ = trace_bfs_dist(*examples[1][:2])

dist = [0, 1, 2, 3, 4]
  cạnh 1->0: dist 1 vs 0 -> giữ | ans=0
  cạnh 1->2: dist 1 vs 2 -> ĐỔI | ans=1
  cạnh 3->2: dist 3 vs 2 -> giữ | ans=1
  cạnh 3->4: dist 3 vs 4 -> ĐỔI | ans=2


## 9. Code nộp LeetCode

Bản khuyên dùng (DFS lặp với cờ chiều), đúng chữ ký `minReorder(self, n: int, connections: List[List[int]]) -> int`:

In [19]:
class Solution:
    def minReorder(self, n: int, connections: List[List[int]]) -> int:
        adj = [[] for _ in range(n)]
        for a, b in connections:
            adj[a].append((b, 1))    # đi a -> b trùng chiều đường: nếu đi ra xa gốc thì phải đổi
            adj[b].append((a, 0))    # chiều ảo
        seen = [False] * n
        seen[0] = True
        stack = [0]
        ans = 0
        while stack:
            u = stack.pop()
            for v, cost in adj[u]:
                if not seen[v]:
                    seen[v] = True
                    ans += cost
                    stack.append(v)
        return ans

for n, c, want in examples:
    assert Solution().minReorder(n, c) == want
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

1. **Cận dưới.** Treo cây ở 0. Xét cạnh nối `p` (cha) và `c` (con). Thành phố `c` chỉ có **một** đường về 0, và đường đó bắt đầu bằng cạnh `c – p`. Nên cạnh này **bắt buộc** phải hướng `c → p`. Mọi cạnh đang hướng `p → c` đều **phải** đổi ⇒ đáp án ≥ số cạnh `cha → con`.
2. **Đủ.** Đổi đúng các cạnh đó thì mọi cạnh đều hướng con → cha; từ bất kỳ `v` đi lên liên tục sẽ tới 0 ⇒ hợp lệ. Vậy đáp án = số cạnh `cha → con`.
3. **DFS đếm đúng số đó.** DFS từ 0 trên đồ thị vô hướng: khi `v` được thăm lần đầu từ `u`, `u` là cha của `v`. Cạnh này mang `cost = 1` khi và chỉ khi nó là cạnh gốc `u → v` (= `cha → con`). Mỗi cạnh cây được duyệt theo chiều cha → con đúng một lần (chiều ngược bị `seen` chặn).
4. **BFS-dist.** Trong cây, cạnh nối cha–con có `dist[con] = dist[cha] + 1`; nên `dist[a] < dist[b]` ⇔ `a` là cha ⇔ cạnh `cha → con`.

## 11. Phân tích Time / Space

Gọi `n` là số thành phố (số cạnh `n − 1`).

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi cách đổi | `O(2^(n−1) · n)` | `O(n)` |
| 2. Quét cạnh lặp lại | `O(n · số vòng)`, xấu nhất `O(n²)` | `O(n)` |
| 3. DFS lặp + cờ | `O(n)` | `O(n)` |
| 4. BFS dist + so sánh | `O(n)` | `O(n)` |

Với `n = 5·10^4`, Cách 3/4 chạy vài chục ms trong Python. DFS **đệ quy** có cùng độ phức tạp nhưng cần tăng `recursionlimit` (rủi ro tràn stack C).

## 12. Bộ test

Đối chiếu brute force trên cây nhỏ ngẫu nhiên (seed cố định), các cách nhanh với nhau trên cây lớn, edge case: `n = 2`, sao (star) hướng vào/ra, chuỗi dài.

In [20]:
def random_tree(n, rng):
    labels = list(range(n))
    rng.shuffle(labels)
    conn = []
    for i in range(1, n):
        p = rng.randrange(i)
        a, b = labels[p], labels[i]
        conn.append([a, b] if rng.random() < 0.5 else [b, a])
    rng.shuffle(conn)
    return conn

FAST = [min_reorder_sweep, min_reorder_dfs, min_reorder_bfs_dist,
        lambda n, c: Solution().minReorder(n, c)]

for n, c, want in examples:
    assert min_reorder_brute(n, c) == want
    for f in FAST:
        assert f(n, c) == want

edge = [
    (2, [[0, 1]], 1),
    (2, [[1, 0]], 0),
    (5, [[0, i] for i in range(1, 5)], 4),          # sao hướng ra
    (5, [[i, 0] for i in range(1, 5)], 0),          # sao hướng vào
    (6, [[i, i + 1] for i in range(5)], 5),          # chuỗi hướng ra xa
]
for n, c, want in edge:
    assert min_reorder_brute(n, c) == want
    for f in FAST:
        assert f(n, c) == want, (n, c)
print("Ví dụ + edge case: OK")

Ví dụ + edge case: OK


In [21]:
rng = random.Random(1466)
for _ in range(300):
    n = rng.randint(2, 9)
    c = random_tree(n, rng)
    want = min_reorder_brute(n, c)
    for f in FAST:
        assert f(n, c) == want, (n, c)
for _ in range(200):
    n = rng.randint(2, 300)
    c = random_tree(n, rng)
    r = [f(n, c) for f in FAST]
    assert len(set(r)) == 1, (n, r)
print("Random: 300 cây nhỏ (so brute) + 200 cây vừa (so chéo): OK")

Random: 300 cây nhỏ (so brute) + 200 cây vừa (so chéo): OK


In [22]:
# Ràng buộc lớn: n = 5*10^4, cây ngẫu nhiên và chuỗi dài
rng = random.Random(99)
big = random_tree(5 * 10**4, rng)
chain = [[i + 1, i] if i % 2 else [i, i + 1] for i in range(5 * 10**4 - 1)]
for name, c in [("cây ngẫu nhiên", big), ("chuỗi dài", chain)]:
    t = time.perf_counter()
    r = Solution().minReorder(5 * 10**4, c)
    dt = time.perf_counter() - t
    assert r == min_reorder_bfs_dist(5 * 10**4, c)
    print(f"{name}: đáp án={r}, {dt:.3f}s")

cây ngẫu nhiên: đáp án=25093, 0.084s


chuỗi dài: đáp án=25000, 0.055s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ đi theo chiều có sẵn của đường

DFS từ 0 chỉ theo `adj[a].append(b)` (đồ thị có hướng) sẽ không tới được những thành phố mà đường đang hướng **về** 0.

In [23]:
def wrong_directed_only(n, connections):
    adj = [[] for _ in range(n)]
    for a, b in connections:
        adj[a].append(b)                         # SAI: chỉ một chiều
    seen = {0}
    stack = [0]
    ans = 0
    while stack:
        u = stack.pop()
        for v in adj[u]:
            if v not in seen:
                seen.add(v)
                ans += 1
                stack.append(v)
    return ans

n, c, want = examples[0]
print("SAI :", wrong_directed_only(n, c), "| đúng:", want)
print("Từ 0 chỉ đi theo chiều đường được 0->1->3 (đếm 2); không tới được 2, 4, 5 nên bỏ sót cạnh 4->5 cần đổi.")

SAI : 2 | đúng: 3
Từ 0 chỉ đi theo chiều đường được 0->1->3 (đếm 2); không tới được 2, 4, 5 nên bỏ sót cạnh 4->5 cần đổi.


**Sửa:** coi đồ thị là vô hướng, ghi cờ chiều gốc.

### Lỗi 2 — Đặt cờ ngược (`cost` cho chiều ảo)

In [24]:
def wrong_flag_swapped(n, connections):
    adj = [[] for _ in range(n)]
    for a, b in connections:
        adj[a].append((b, 0))                    # SAI: lẽ ra là 1
        adj[b].append((a, 1))                    # SAI: lẽ ra là 0
    seen = [False] * n; seen[0] = True
    stack, ans = [0], 0
    while stack:
        u = stack.pop()
        for v, cost in adj[u]:
            if not seen[v]:
                seen[v] = True; ans += cost; stack.append(v)
    return ans

n, c, want = examples[0]
print("SAI :", wrong_flag_swapped(n, c), "| đúng:", want, "| (n-1) - đúng =", n - 1 - want)

SAI : 2 | đúng: 3 | (n-1) - đúng = 2


Đặt cờ ngược sẽ đếm số cạnh **đúng chiều**, tức `(n − 1) − đáp án`. **Sửa:** cạnh gốc `a → b` có cost 1 khi đi từ `a` sang `b`.

### Lỗi 3 — Quên đánh dấu `visited` trên đồ thị vô hướng

Từ `u` sang `v`, rồi từ `v` lại quay về `u`… vòng lặp không dừng. (Cell dưới đặt giới hạn bước để không treo.)

In [25]:
def wrong_no_visited(n, connections, limit=1000):
    adj = build_adj(n, connections)
    stack, ans, steps = [0], 0, 0
    while stack:
        steps += 1
        if steps > limit:
            return f"không dừng (vượt {limit} bước), ans đang là {ans}"
        u = stack.pop()
        for v, cost in adj[u]:                   # SAI: không kiểm tra đã thăm
            ans += cost
            stack.append(v)
    return ans

n, c, want = examples[2]
print("SAI :", wrong_no_visited(n, c), "| đúng:", want)

SAI : không dừng (vượt 1000 bước), ans đang là 500 | đúng: 0


**Sửa:** mảng `seen` (hoặc truyền `parent` trong DFS và bỏ qua `v == parent`).

### Lỗi 4 — DFS đệ quy không tăng giới hạn trên cây sâu

Đúng về logic nhưng chuỗi dài làm vượt giới hạn đệ quy (ở đây bắt lỗi để in ra).

In [26]:
def wrong_recursive(n, connections):
    adj = build_adj(n, connections)
    seen = [False] * n
    def dfs(u):
        seen[u] = True
        total = 0
        for v, cost in adj[u]:
            if not seen[v]:
                total += cost + dfs(v)
        return total
    return dfs(0)

n = 20000
c = [[i, i + 1] for i in range(n - 1)]
try:
    print("kết quả:", wrong_recursive(n, c))
except RecursionError:
    print("SAI : RecursionError với chuỗi", n, "đỉnh | đúng:", min_reorder_dfs(n, c))

SAI : RecursionError với chuỗi 20000 đỉnh | đúng: 19999


**Sửa:** DFS lặp bằng stack / BFS (hoặc `sys.setrecursionlimit` lớn — kém an toàn hơn).

### Lỗi 5 — Chỉ quan tâm các đường nối trực tiếp với 0

Nghĩ rằng chỉ cần đổi các đường xuất phát từ 0 (`[0, x]`).

In [27]:
def wrong_only_root_edges(n, connections):
    return sum(a == 0 for a, b in connections)   # SAI

n, c, want = examples[0]
print("SAI :", wrong_only_root_edges(n, c), "| đúng:", want)
n, c, want = examples[1]
print("SAI :", wrong_only_root_edges(n, c), "| đúng:", want)

SAI : 1 | đúng: 3
SAI : 0 | đúng: 2


**Sửa:** mọi cạnh ở mọi độ sâu đều phải hướng về gốc — duyệt toàn cây.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- `n` đỉnh, `n − 1` cạnh, "chỉ có một đường giữa hai đỉnh" ⇒ **cây** ⇒ treo ở gốc, mỗi cạnh là cha–con.
- Đồ thị có hướng nhưng câu hỏi về "đổi chiều" ⇒ **xây vô hướng kèm trọng số 0/1** (cạnh thật 1, cạnh ảo 0 hoặc ngược lại tùy câu hỏi).
- Đồ thị **tổng quát** (không phải cây) với câu hỏi "ít cạnh đảo nhất để đi từ s tới t" ⇒ cùng mẹo trọng số 0/1 + **BFS 0-1** (bài 1368 / "Minimum Edge Reversals").
- `n ≤ 10^5` trong Python ⇒ ưu tiên **DFS lặp / BFS**, tránh đệ quy.
- Muốn đáp án cho **mọi gốc** ⇒ **rerooting**: đổi gốc qua một cạnh chỉ thay đổi đóng góp của chính cạnh đó.

## 15. Biến thể của bài

1. **Thủ đô bất kỳ `t`** thay vì 0: chạy DFS từ `t` (bài tập 4).
2. **Đáp án cho mọi thủ đô** (LeetCode 2858 — Minimum Edge Reversals So Every Node Is Reachable, chiều ngược lại): rerooting `O(n)` (bài tập 5).
3. **Liệt kê các cạnh cần đổi** (bài tập 3).
4. **Mọi thành phố đi được TỪ 0** (thay vì tới 0): đáp án = số cạnh `con → cha` = `(n − 1) −` đáp án gốc.
5. **Đồ thị tổng quát, ít cạnh đảo nhất từ `s` tới `t`:** BFS 0-1 / Dijkstra với trọng số 0 (đi xuôi) và 1 (đi ngược).

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi cách đổi | `2^(n−1)` cấu hình, kiểm tra từng cái | `O(2^n · n)` | `O(n)` | Kiểm chứng `n` nhỏ |
| 2. Quét cạnh lặp | Lan dần tập "về được 0" | `O(n²)` xấu nhất | `O(n)` | Trực giác, input dễ |
| 3. DFS + cờ 0/1 | Đếm cạnh cha → con khi đi từ 0 | `O(n)` | `O(n)` | Mặc định |
| 4. BFS dist | Cạnh `a→b` đổi nếu `dist[a] < dist[b]` | `O(n)` | `O(n)` | Cần cấu trúc cây tường minh |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `minReorder(4, [[1,0],[1,2],[3,2]])` = ?

**Bài 2 (tính tay):** Với ví dụ 2, lập bảng `dist` và chỉ ra cạnh nào phải đổi.

**Bài 3 (điền TODO):** Viết `reversed_edges(n, connections)` trả về **danh sách chỉ số** `i` (tăng dần) của các cạnh cần đổi chiều.

**Bài 4 (tự viết lại):** Viết `min_reorder_to(n, connections, t)`: số cạnh tối thiểu cần đổi để mọi thành phố tới được `t`.

**Bài 5 (mini challenge):** Viết `min_reorder_all(n, connections)` trả về mảng `ans[t]` cho **mọi** `t` trong `O(n)` tổng cộng (rerooting).

In [28]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def reversed_edges(n: int, connections: List[List[int]]) -> List[int]:
    # TODO: BFS dist từ 0, rồi chọn i có dist[a] < dist[b]
    raise NotImplementedError

# Bài 4 — TODO
def min_reorder_to(n: int, connections: List[List[int]], t: int) -> int:
    # TODO: giống Cách 3 nhưng bắt đầu từ t
    raise NotImplementedError

# Bài 5 — TODO
def min_reorder_all(n: int, connections: List[List[int]]) -> List[int]:
    # TODO: ans[0] bằng DFS; đi từ cha u sang con v qua cạnh có cost c:
    #       ans[v] = ans[u] - 1 nếu c == 1 (cạnh u->v) , ans[u] + 1 nếu c == 0 (cạnh v->u)
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Vẽ cây: `0 – 1 – 2 – 3`. (2) `1→0` đúng, `1→2` hướng ra xa. (3) `3→2` đúng ⇒ chỉ đổi 1 cạnh.

**Bài 2.** (1) BFS từ 0: `dist[1] = 1`. (2) `dist[2] = 2`, `dist[3] = 3`, `dist[4] = 4`. (3) Cạnh `1→2` (`1 < 2`) và `3→4` (`3 < 4`) phải đổi.

**Bài 3.** (1) Dùng lại `min_reorder_bfs_dist` nhưng giữ `dist`. (2) `enumerate(connections)`. (3) `[i for i, (a, b) in enumerate(connections) if dist[a] < dist[b]]`.

**Bài 4.** (1) Chỉ khác điểm xuất phát. (2) `seen[t] = True`, `stack = [t]`. (3) Kiểm tra bằng cách đổi nhãn `t ↔ 0` rồi gọi lời giải gốc.

**Bài 5.** (1) Tính `ans[0]` như Cách 3. (2) Khi đổi gốc từ `u` sang con `v`, chỉ cạnh `u – v` thay đổi vai trò: cạnh `u → v` trước phải đổi, nay đúng chiều (`−1`); cạnh `v → u` trước đúng, nay phải đổi (`+1`). (3) Duyệt DFS từ 0 lần hai để lan `ans`.

## 19. Đáp án tham khảo cho bài tập

In [29]:
# Bài 1, Bài 2
print("Bài 1:", min_reorder_dfs(4, [[1, 0], [1, 2], [3, 2]]))
assert min_reorder_dfs(4, [[1, 0], [1, 2], [3, 2]]) == 1

def bfs_dist(n, connections, root=0):
    adj = [[] for _ in range(n)]
    for a, b in connections:
        adj[a].append(b); adj[b].append(a)
    dist = [-1] * n; dist[root] = 0
    q = deque([root])
    while q:
        u = q.popleft()
        for v in adj[u]:
            if dist[v] == -1:
                dist[v] = dist[u] + 1; q.append(v)
    return dist

d2 = bfs_dist(*examples[1][:2])
print("Bài 2: dist =", d2)
assert d2 == [0, 1, 2, 3, 4]

Bài 1: 1
Bài 2: dist = [0, 1, 2, 3, 4]


In [30]:
def reversed_edges_ans(n, connections):
    dist = bfs_dist(n, connections)
    return [i for i, (a, b) in enumerate(connections) if dist[a] < dist[b]]

def min_reorder_to_ans(n, connections, t):
    adj = build_adj(n, connections)
    seen = [False] * n; seen[t] = True
    stack, ans = [t], 0
    while stack:
        u = stack.pop()
        for v, cost in adj[u]:
            if not seen[v]:
                seen[v] = True; ans += cost; stack.append(v)
    return ans

assert reversed_edges_ans(*examples[0][:2]) == [0, 1, 4]
assert reversed_edges_ans(*examples[1][:2]) == [1, 3]
rng = random.Random(4)
for _ in range(300):
    n = rng.randint(2, 30)
    c = random_tree(n, rng)
    assert len(reversed_edges_ans(n, c)) == min_reorder_dfs(n, c)
    t = rng.randrange(n)
    swap = lambda x: 0 if x == t else (t if x == 0 else x)      # đổi nhãn t <-> 0
    c2 = [[swap(a), swap(b)] for a, b in c]
    assert min_reorder_to_ans(n, c, t) == min_reorder_dfs(n, c2)
print("Bài 3, 4: OK", reversed_edges_ans(*examples[0][:2]))

Bài 3, 4: OK [0, 1, 4]


In [31]:
def min_reorder_all_ans(n, connections):
    adj = build_adj(n, connections)
    ans = [0] * n
    ans[0] = min_reorder_dfs(n, connections)
    seen = [False] * n; seen[0] = True
    stack = [0]
    while stack:
        u = stack.pop()
        for v, cost in adj[u]:
            if not seen[v]:
                seen[v] = True
                ans[v] = ans[u] - 1 if cost == 1 else ans[u] + 1
                stack.append(v)
    return ans

rng = random.Random(12)
for _ in range(200):
    n = rng.randint(2, 25)
    c = random_tree(n, rng)
    assert min_reorder_all_ans(n, c) == [min_reorder_to_ans(n, c, t) for t in range(n)]
print("Bài 5: ví dụ 1 ->", min_reorder_all_ans(*examples[0][:2]), "| OK 200 cây")

Bài 5: ví dụ 1 -> [3, 2, 2, 1, 4, 3] | OK 200 cây


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "cây có hướng → đếm cạnh hướng sai so với gốc":**

```python
adj = [[] for _ in range(n)]
for a, b in connections:
    adj[a].append((b, 1))      # cạnh thật
    adj[b].append((a, 0))      # cạnh ảo
seen = [False] * n; seen[0] = True
stack, ans = [0], 0
while stack:
    u = stack.pop()
    for v, cost in adj[u]:
        if not seen[v]:
            seen[v] = True
            ans += cost
            stack.append(v)
return ans
```

**Checklist:**
- [ ] Đã thêm cạnh vào **cả hai** đầu (vô hướng) kèm cờ chiều.
- [ ] Cạnh thật `a → b` mang cost 1 khi đi `a → b` (ra xa gốc).
- [ ] Có `seen` / `parent` để không quay lại.
- [ ] Không dùng đệ quy sâu (hoặc đã tăng giới hạn).
- [ ] Đã thử: `n = 2` hai chiều, sao vào/ra, chuỗi dài.